# Lab 01 — Autonomous Issue Triage Agent
### Fix It Fast: Autonomous Bug Resolution with Claude Code on Amazon Bedrock AgentCore

In this lab you will watch Claude Code autonomously triage a GitHub issue:
1. **Read** the issue title and body
2. **Classify** its type (`bug`, `feature`, `question`, `duplicate`) and severity (`P0`–`P3`)
3. **Label** the issue with type, severity, and `agent:claude-code`
4. **Comment** with a structured triage summary — and ask for repro steps if they're missing

The agent never closes an issue or merges anything — it only prepares the issue for a human developer to pick up.

**Architecture**

```
Your machine
  └─ connect.py ──(SigV4 WebSocket)──► AgentCore Runtime (microVM)
                                            └─ Claude Code
                                                  ├─ Amazon Bedrock  (model inference)
                                                  └─ AgentCore Gateway  (GitHub MCP tools)
                                                        └─ GitHub API
```

---
### Prerequisites

| Requirement | Check |
|---|---|
| AWS credentials | `aws sts get-caller-identity` |
| Python 3.10+ | `python3 --version` |
| `gh` CLI authenticated | `gh auth status` |
| `lab-00-deploy.ipynb` completed (or instructor pre-deployed) | `runtime_config.json` exists |

## Step 1 — Configuration

In [ ]:
import os, subprocess, json, sys, re

WORKSHOP_DIR = os.path.abspath("")
AGENT        = "claude-code"   # ← swap to "codex" or another agent folder name

# ── Load config written by lab-00 (override any value below if needed) ────────
_cfg_path = os.path.join(WORKSHOP_DIR, "workshop_config.json")
if os.path.exists(_cfg_path):
    _cfg = json.load(open(_cfg_path))
    GITHUB_USER = _cfg["GITHUB_USER"]
    GITHUB_REPO = _cfg["GITHUB_REPO"]
    AWS_REGION  = _cfg["AWS_REGION"]
    AWS_PROFILE = _cfg["AWS_PROFILE"]
    print("Loaded config from workshop_config.json")
else:
    GITHUB_USER = "your-github-username"
    GITHUB_REPO = "my-task-manager"
    AWS_REGION  = "us-west-2"
    AWS_PROFILE = ""  # set to your AWS profile name, or leave empty to use default
    print("workshop_config.json not found — using defaults above")
# ─────────────────────────────────────────────────────────────────────────────

os.environ["AWS_REGION"]         = AWS_REGION
os.environ["AWS_DEFAULT_REGION"] = AWS_REGION
os.environ["AWS_PROFILE"]        = AWS_PROFILE
os.environ["NO_COLOR"]           = "1"

AGENT_DIR = os.path.join(WORKSHOP_DIR, "coding_agents", AGENT)

_ANSI = re.compile(r"\x1b\[[0-9;]*m")

def run(cmd, cwd=None, capture=False):
    env = os.environ.copy()
    if capture:
        r = subprocess.run(cmd, shell=True, cwd=cwd, env=env,
                           stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, text=True)
        r.check_returncode()
        return _ANSI.sub("", r.stdout).strip()
    subprocess.run(cmd, shell=True, cwd=cwd, env=env, check=True)

print(f"Region:  {AWS_REGION}")
print(f"Profile: {AWS_PROFILE}")
print(f"Repo:    {GITHUB_USER}/{GITHUB_REPO}")
print(f"Agent:   {AGENT}")

## Step 2 — Verify credentials and runtime

In [ ]:
identity = json.loads(run("aws sts get-caller-identity --output json", capture=True))
print(f"AWS Account: {identity['Account']}")
print(f"Identity:    {identity['Arn']}")

with open(os.path.join(AGENT_DIR, "runtime_config.json")) as f:
    RUNTIME_ARN = json.load(f)["runtime_arn"]

print(f"\nRuntime ARN: {RUNTIME_ARN}")
print("\nReady.")

## Step 3 — Install dependencies

In [ ]:
import sys
run(f"{sys.executable} -m pip install -q -r {os.path.join(WORKSHOP_DIR, 'coding_agents', 'requirements.txt')}")
print("Done.")

## Step 4 — Browse the open issues

The repository has 9 open issues seeded by `lab-00`. Each represents a different bug type. Read through them and pick one to triage.

In [ ]:
issues_raw = run(
    f"gh issue list --repo {GITHUB_USER}/{GITHUB_REPO} --state open "
    f"--json number,title,body,labels --limit 20",
    capture=True
)
issues = json.loads(issues_raw)

print(f"Open issues in {GITHUB_USER}/{GITHUB_REPO}:\n")
for i in issues:
    labels = ", ".join(l["name"] for l in i.get("labels", []))
    label_str = f"  [{labels}]" if labels else ""
    print(f"  #{i['number']:2d}  {i['title']}{label_str}")
    if i.get("body"):
        print(f"       {i['body'].splitlines()[0]}")
    print()

print(f"Total: {len(issues)} open issues")

## Step 5 — Pick an issue and run the triage agent

Change `TARGET_ISSUE` to any number from the list above, then run the next cell.

The agent will:
- Read the issue title and body via GitHub MCP tools
- Classify the type: `bug`, `feature`, `question`, or `duplicate`
- Assign a severity: `P0` (critical) → `P3` (low)
- Apply labels: type + severity + `agent:claude-code`
- Post a structured triage comment; ask for repro steps if missing

**Watch the output — you'll see every tool call in real time.**

In [ ]:
TARGET_ISSUE = issues[0]["number"]  # ← change this to any issue number

title = next(i["title"] for i in issues if i["number"] == TARGET_ISSUE)
print(f"Triaging issue #{TARGET_ISSUE}: {title}\n")

In [ ]:
TRIAGE_INSTRUCTIONS = """
You are a triage agent. Triage the following GitHub issue:

  Repository: {owner}/{repo}
  Issue: #{number}

Steps:
1. Read the issue using get_issue.
2. Classify the issue type: bug | feature | question | duplicate.
3. Assign a severity based on impact:
     P0 = production down / data loss
     P1 = major functionality broken
     P2 = minor functionality affected
     P3 = cosmetic / low impact
4. Apply labels via add_labels: the type label, the severity label (e.g. "P2"), and "agent:claude-code".
   Create labels if they don't exist (the API will handle that automatically).
5. Post a comment via comment_on_issue with this structure:
   ## Triage Summary
   **Type:** <type>
   **Severity:** <P0-P3> -- <one-sentence reason>
   **Assessment:** <2-3 sentences describing the bug/impact>
   **Repro steps provided:** Yes / No
   <If no repro steps: add a section asking the reporter to provide them>
   **Next step:** Ready for a developer to pick up.

Do not create branches, commits, or PRs. Do not close the issue.
""".format(owner=GITHUB_USER, repo=GITHUB_REPO, number=TARGET_ISSUE)

print(f"Prompt sent to agent:\n{TRIAGE_INSTRUCTIONS}")
print("=" * 60)

run(f'{sys.executable} connect.py --prompt "{TRIAGE_INSTRUCTIONS.strip()}"', cwd=AGENT_DIR)

## Step 6 — Verify the triage result

Check what the agent did: labels applied and the comment it left.

In [ ]:
issue = json.loads(run(
    f"gh issue view {TARGET_ISSUE} --repo {GITHUB_USER}/{GITHUB_REPO} --json number,title,labels,comments",
    capture=True
))

labels = [l["name"] for l in issue.get("labels", [])]
print(f"Issue #{issue['number']}: {issue['title']}")
print(f"Labels: {labels}\n")

comments = issue.get("comments", [])
if comments:
    print("Latest comment:")
    print("-" * 40)
    print(comments[-1]["body"])
else:
    print("No comments yet.")

## Step 7 — Triage all open issues in one go

Run the triage agent on every open issue sequentially. This is how you'd process a backlog.

In [ ]:
untriaged = [
    i for i in issues
    if not any(l["name"] in ("bug", "feature", "question", "duplicate") for l in i.get("labels", []))
]

print(f"Untriaged issues: {len(untriaged)}\n")

for issue in untriaged:
    n, title = issue["number"], issue["title"]
    print(f"\n{'='*60}\nIssue #{n}: {title}\n{'='*60}")
    prompt = """
You are a triage agent. Triage GitHub issue #{n} in {owner}/{repo}.
1. Read the issue. 2. Classify type (bug/feature/question/duplicate) and severity (P0-P3).
3. Apply labels: type + severity + agent:claude-code.
4. Post a triage comment with type, severity, assessment, and whether repro steps are provided.
Do not create branches, commits, or PRs. Do not close the issue.
""".format(n=n, owner=GITHUB_USER, repo=GITHUB_REPO)
    run(f'{sys.executable} connect.py --prompt "{prompt.strip()}"', cwd=AGENT_DIR)

print("\nAll issues triaged.")

## Step 8 — Summary

In [ ]:
all_issues = json.loads(run(
    f"gh issue list --repo {GITHUB_USER}/{GITHUB_REPO} --state open "
    f"--json number,title,labels --limit 20",
    capture=True
))

triaged = [
    i for i in all_issues
    if any(l["name"] in ("bug", "feature", "question", "duplicate") for l in i.get("labels", []))
]

print(f"Total open issues: {len(all_issues)}")
print(f"Triaged:           {len(triaged)}")
print()
for i in triaged:
    labels = ", ".join(l["name"] for l in i["labels"])
    print(f"  #{i['number']:2d}  {i['title']}")
    print(f"       Labels: {labels}")

## Step 9 — What's next?

The triage agent has classified and labelled every issue. Now a developer (or another agent) can filter by `bug` + `P0`/`P1` and start fixing.

**Continue to `lab-02-bug-fix-agent.ipynb`** — run the bug-fix agent on any issue that was labelled `bug` by the triage agent.

---
## Key Takeaways

- **Same runtime, different behaviour** — triage and bug-fix use the same container; the prompt drives the task
- **CLAUDE.md as a base layer** — sets tool conventions and guardrails; the prompt adds task-specific instructions on top
- **Agent as a pipeline stage** — triage output (labels) becomes the filter input for the next agent (bug-fix)
- **AgentCore Gateway** — GitHub is exposed as IAM-authenticated MCP tools; no API keys in the container